In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math

# Single Dimensional Mean Estimation



In [ ]:
# Algorithm 1: Laplace Mechanism for Heterogenous LDP
def lap_mech_1D(X, epsilons, calculate_weights):
  N = len(X)

  noise = np.random.laplace(loc=0, scale=2/epsilons, size=N)
  Y = X + noise

  weights = calculate_weights(epsilons)

  return np.dot(weights, Y)

# Algorithm 2: Randomized Response Mechanism for Heterogenous LDP
def RR_1D(X, epsilons, calc_opt_weights, calc_unif_weights):
    N = len(X)

    # Run randomzied response based on eps_i
    p = np.exp(epsilons) / (np.exp(epsilons) + 1)
    signs = np.where(np.random.rand(N) < p, 1, -1)
    Y = signs * X

    C = (np.exp(epsilons) + 1) / (np.exp(epsilons) - 1)
    opt_weights = calc_opt_weights(C)
    unif_weights = calc_unif_weights(epsilons)

    # Weighted average of noisy data
    return np.dot(opt_weights * C, Y), np.dot(unif_weights * C, Y)

# Calculate weights
def calc_lap_mech_optimal_weights(epsilons):
  weights = (1 / (1 + (1 / (epsilons ** 2))))
  weights /= weights.sum()
  return weights

def calc_RR_optimal_weights(C):
  weights = 1 / (C ** 2)
  weights /= weights.sum()
  return weights

def calc_uniform_weights(epsilons):
  weights = np.ones((len(epsilons),))
  weights /= len(epsilons)
  return weights


# Generate epsilons where an alpha proportion of users have epsilon = 0.1
def generate_epsilons(N, alpha):
  epsilons = np.ones(N,)

  for i in range(int(N * alpha)):
    epsilons[i] = 0.1

  return epsilons

# Lower bound distribution
def calc_theta_LB_dist_1D(beta, epsilons):

  term = np.log(1 / ((4 * beta) * (1 -  beta))) / (8 * (epsilons ** 2).sum())
  theta_squared = min(term, 1)

  return np.sqrt(theta_squared)

def sample_from_LB_dist_1D(theta, N):
  p = (1 + theta) / 2
  return np.where(np.random.rand(N) < p, 1, -1)

In [ ]:
def run_RR_1D_experiment(N, alpha, beta, num_of_trials):
  epsilons = generate_epsilons(N, alpha)
  theta = calc_theta_LB_dist_1D(beta, epsilons)
  samples = sample_from_LB_dist_1D(theta, N)

  trials = range(num_of_trials)
  mse_opt = []
  mse_unif = []

  for i in trials:
    optimal_estimate, uniform_estimate = RR_1D(samples, epsilons, calc_RR_optimal_weights, calc_uniform_weights)

    mse_opt.append((optimal_estimate - theta) ** 2)
    mse_unif.append((uniform_estimate - theta) ** 2)

  avg_optimal_mse = sum(mse_opt) / num_of_trials
  avg_uniform_mse = sum(mse_unif) / num_of_trials

  return mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse

def plot_RR_1D_experiment(mse_opt, mse_unif, N, alpha, num_of_trials):
  trials = range(num_of_trials)

  plt.figure()
  plt.plot(trials, mse_opt, label="Optimal Weights")
  plt.plot(trials, mse_unif, label="Uniform Weights")

  plt.xlabel("Trials")
  plt.ylabel("MSE")
  plt.title(f"MSE of 1D RR Estimates with alpha = {alpha}, n = {N}")
  plt.legend()
  plt.grid(True)

In the single-dimensional case, we run experiments using Algorithm 2 which relies on randomized response. We compare the mean estimate produced by our algorithm using optimal weights against the mean estimate under uniform weights. We randomly select N samples from the distributions described in the proof of our lower bounds. Information theoretically, these are the most difficult data distributions to deal with. We assign an α fraction of users to ɛᵢ = 0.1 and the remaining (1 - α ) users to ɛᵢ = 1. We report the average MSE across 100 trials.

First, we consider multiple sample sizes: 10, 100, 1000, 10000, 100000.

Then, we consider different values of α: 0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.

In [ ]:
TRIALS = 100
BETA = 0.01
ALPHA = 0.5 # Prop. of users with eps = 0.1; (1 - alpha) users have eps = 1
N = 1000

# Experiment 1: Vary sample size
sample_sizes = [10, 100, 1000, 10000, 100000]
avg_opt_mses_ss =[]
avg_unif_mses_ss = []

for n in sample_sizes:
  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_RR_1D_experiment(n, ALPHA, BETA, TRIALS)
  # plot_RR_1D_experiment(mse_opt, mse_unif, n, ALPHA, TRIALS)

  avg_opt_mses_ss.append(avg_optimal_mse)
  avg_unif_mses_ss.append(avg_uniform_mse)

print(f"Fix alpha = {ALPHA}, trials = {TRIALS}, beta = {BETA}.")
mse_sample_size_df = pd.DataFrame({
    "Sample Size": sample_sizes,
    "Optimal": avg_opt_mses_ss,
    "Uniform": avg_unif_mses_ss,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_ss, avg_opt_mses_ss)],
})
print(mse_sample_size_df)


Fix alpha = 0.5, trials = 100, beta = 0.01.
   Sample Size   Optimal    Uniform  Uniform - Optimal
0           10  0.631969  22.573545          21.941576
1          100  0.094048   1.938500           1.844452
2         1000  0.008707   0.210858           0.202151
3        10000  0.000949   0.014844           0.013895
4       100000  0.000069   0.001448           0.001379


In [ ]:
# Experiment 2: Vary alpha
N = 1000
alphas = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
avg_opt_mses_a =[]
avg_unif_mses_a = []

for alpha in alphas:
  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_RR_1D_experiment(N, alpha, BETA, TRIALS)
  # plot_RR_1D_experiment(mse_opt, mse_unif, N, alpha, TRIALS)

  avg_opt_mses_a.append(avg_optimal_mse)
  avg_unif_mses_a.append(avg_uniform_mse)

print(f"Fix n = {N}, trials = {TRIALS}, beta = {BETA}.")
mse_alpha_df = pd.DataFrame({
    "Alpha": alphas,
    "Optimal": avg_opt_mses_a,
    "Uniform": avg_unif_mses_a,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_a, avg_opt_mses_a)],
})
print(mse_alpha_df)

Fix n = 1000, trials = 100, beta = 0.01.
    Alpha   Optimal   Uniform  Uniform - Optimal
0     0.0  0.002641  0.002641           0.000000
1     0.1  0.006793  0.038965           0.032172
2     0.2  0.005680  0.078926           0.073245
3     0.3  0.006692  0.107954           0.101262
4     0.4  0.010832  0.224190           0.213358
5     0.5  0.005313  0.205487           0.200173
6     0.6  0.014259  0.296116           0.281857
7     0.7  0.014865  0.260656           0.245791
8     0.8  0.034397  0.286146           0.251749
9     0.9  0.049770  0.379657           0.329887
10    1.0  0.426827  0.426827           0.000000


# Multi-Dimensional Mean Estimation

In the multi-dimensional case, we run experiments using Algorithm 3. We compare the mean estimate produced by our algorithm using optimal weights against the mean estimate under uniform weights. We randomly select N samples from the distributions described in the proof of our lower bounds. Information theoretically, these are the most difficult data distributions to deal with. We assign an α fraction of users to ɛᵢ = 0.1 and the remaining (1 - α ) users to ɛᵢ = 1. We report the average MSE across 100 trials varying different parameters in each experiment.

In [ ]:
# Algorithm 4: LDP Randomizer of Duchi
def duchi_randomizer(X_i, eps_i, r):
  d = len(X_i)

  norm = np.linalg.norm(X_i)
  p_sign = 1/2 + (norm / (2 * r))
  sign = 1 if np.random.rand() < p_sign else -1

  X_tilde = (sign * r / norm) * X_i

  p_T = np.exp(eps_i) / (np.exp(eps_i) + 1)
  T = np.random.binomial(n=1, p=p_T)

  c_i = (np.exp(eps_i) + 1) / (np.exp(eps_i) - 1)

  B_i = ( c_i * r * d * np.sqrt(np.pi) ) * (math.gamma(((d - 1) / 2) + 1) / math.gamma(d/2 + 1))


  # Sample uniformly from sphere with radius B_i
  Y_i = np.random.randn(d)
  Y_i /= np.linalg.norm(Y_i)
  Y_i *= B_i

  dot_YX = np.dot(Y_i, X_tilde)

  # Enforce hemisphere constraint
  if (T and dot_YX <= 0) or (not T and dot_YX > 0):
    Y_i *= -1

  return Y_i

# Algorithm 3: Optimal Multi-D Mechanism for Heterogenous LDP
def multi_d_mean_est(X, epsilons, r, calc_opt_weights, calc_unif_weights):
  N = X.shape[0]
  Y = np.zeros_like(X)

  for i in range(N):
    Y[i] = duchi_randomizer(X[i], epsilons[i], r)

  opt_weights = calc_opt_weights(epsilons)
  unif_weights = calc_unif_weights(epsilons)

  return np.dot(opt_weights, Y), np.dot(unif_weights, Y)

# Calculate optimal weights
def calc_multi_d_optimal_weights(epsilons):
  weights = (epsilons ** 2)
  weights /= weights.sum()
  return weights

# Lower bound distribution
def calc_psi_LB_dist_multi_D(k, epsilons):
  term = (k ** 2) / (16 * (epsilons ** 2).sum() )
  psi_squared = min(term, 1)

  return np.sqrt(psi_squared)

def sample_from_LB_dist_multi_D(psi, r, N, d, k, nu):
  js = np.random.randint(0, k, size=N)   # Sample index j for each user

  p = (1 + psi * nu[js]) / 2   # Probability based on nu[j] for each user
  signs = np.where(np.random.rand(N) < p, 1, -1)

  X = np.zeros((N, d))

  for i, j in enumerate(js):
    X[i, j] = r * signs[i]

  return X

In [ ]:
def run_multi_D_experiment(N, epsilons, beta, num_of_trials, d, r, k):


  psi = calc_psi_LB_dist_multi_D(k, epsilons)
  nu = np.where(np.random.rand(k) < 0.5, 1, -1)
  samples = sample_from_LB_dist_multi_D(psi, r, N, d, k, nu)
  theta = (psi * r * nu) / k


  trials = range(num_of_trials)
  mse_opt = []
  mse_unif = []

  for i in trials:
    optimal_estimate, uniform_estimate  = multi_d_mean_est(samples, epsilons, r, calc_multi_d_optimal_weights, calc_uniform_weights)
    mse_opt.append( np.mean((optimal_estimate[:k] - theta)**2) )
    mse_unif.append( np.mean((uniform_estimate[:k] - theta)**2) )

  avg_optimal_mse = sum(mse_opt) / num_of_trials
  avg_uniform_mse = sum(mse_unif) / num_of_trials

  return mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse

def plot_multi_D_experiment(mse_opt, mse_unif, N, alpha, num_of_trials):
  trials = range(num_of_trials)

  plt.figure()
  plt.plot(trials, mse_opt, label="Optimal Weights")
  plt.plot(trials, mse_unif, label="Uniform Weights")

  plt.xlabel("Trials")
  plt.ylabel("MSE")
  plt.title(f"MSE of Multi D Estimates with alpha = {alpha}, n = {N}")
  plt.legend()
  plt.grid(True)

In [ ]:
TRIALS = 100
BETA = 0.01
N = 1000
ALPHA = 0.5
D = 10
K = D
R = 1


# Experiment 1: Vary sample size
sample_sizes = [10, 100, 1000]
avg_opt_mses_ss = []
avg_unif_mses_ss = []

for n in sample_sizes:
  epsilons = generate_epsilons(n, ALPHA)

  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_multi_D_experiment(n, epsilons, BETA, TRIALS, D, R, K)
  # plot_multi_D_experiment(mse_opt, mse_unif, n, ALPHA, TRIALS)

  avg_opt_mses_ss.append(avg_optimal_mse)
  avg_unif_mses_ss.append(avg_uniform_mse)

print(f"Fix alpha = {ALPHA}, d = {D}, k = {K}")
mse_sample_size_df = pd.DataFrame({
    "Sample Size": sample_sizes,
    "Optimal": avg_opt_mses_ss,
    "Uniform": avg_unif_mses_ss,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_ss, avg_opt_mses_ss)],
})
print(mse_sample_size_df)

Fix alpha = 0.5, d = 10, k = 10
   Sample Size   Optimal     Uniform  Uniform - Optimal
0           10  5.718696  119.013652         113.294955
1          100  0.575147   12.612270          12.037123
2         1000  0.054144    1.259144           1.205000


In [ ]:
# Experiment 2: Vary alpha
alphas = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
D = 10
avg_opt_mses_a = []
avg_unif_mses_a = []

for alpha in alphas:
  epsilons = generate_epsilons(N, alpha)

  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_multi_D_experiment(N, epsilons, BETA, TRIALS, D, R, D)
  # plot_multi_D_experiment(mse_opt, mse_unif, N, alpha, TRIALS)

  avg_opt_mses_a.append(avg_optimal_mse)
  avg_unif_mses_a.append(avg_uniform_mse)


print(f"Fix N = {N}, d = {D}, k = {D}.")
mse_alpha_df = pd.DataFrame({
    "Alpha": alphas,
    "Optimal": avg_opt_mses_a,
    "Uniform": avg_unif_mses_a,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_a, avg_opt_mses_a)],
})
print(mse_alpha_df)

Fix N = 1000, d = 10, k = 10.
    Alpha   Optimal   Uniform  Uniform - Optimal
0     0.0  0.030004  0.030004       0.000000e+00
1     0.1  0.032829  0.260186       2.273564e-01
2     0.2  0.034123  0.525625       4.915018e-01
3     0.3  0.039052  0.788690       7.496389e-01
4     0.4  0.047224  0.973778       9.265542e-01
5     0.5  0.053988  1.189010       1.135021e+00
6     0.6  0.069302  1.459746       1.390445e+00
7     0.7  0.101638  1.666755       1.565117e+00
8     0.8  0.120370  2.053585       1.933215e+00
9     0.9  0.265863  2.093345       1.827482e+00
10    1.0  2.492449  2.492449       2.664535e-15


In [ ]:
# Experiment 3: Vary dimension, let d = k
dims = [2, 10, 100, 250]
avg_opt_mses_d = []
avg_unif_mses_d = []

for d in dims:
  epsilons = generate_epsilons(N, ALPHA)

  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_multi_D_experiment(N, epsilons, BETA, TRIALS, d, R, d)
  # plot_multi_D_experiment(mse_opt, mse_unif, N, ALPHA, TRIALS)

  avg_opt_mses_d.append(avg_optimal_mse)
  avg_unif_mses_d.append(avg_uniform_mse)

print(f"Fix N = {N}, alpha = {ALPHA}, k = d.")
mse_d_df = pd.DataFrame({
    "d": dims,
    "Optimal": avg_opt_mses_d,
    "Uniform": avg_unif_mses_d,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_d, avg_opt_mses_d)],
})
print(mse_d_df)

Fix N = 1000, alpha = 0.5, k = d.
     d   Optimal   Uniform  Uniform - Optimal
0    2  0.045523  0.999600           0.954077
1   10  0.058093  1.113305           1.055212
2  100  0.057248  1.267225           1.209977
3  250  0.057698  1.277179           1.219482


In [ ]:
# Experiment 4: Vary k
D = 100
ks = [1, D//2, D]
k_e = int(4 * np.sqrt((epsilons ** 2).sum()))
if k_e < D:
  ks.append(k_e)
  ks.sort()
epsilons = generate_epsilons(N, ALPHA)

avg_opt_mses_k = []
avg_unif_mses_k = []


for k in ks:
  mse_opt, mse_unif, avg_optimal_mse, avg_uniform_mse = run_multi_D_experiment(N, epsilons, BETA, TRIALS, D, R, k)
  # plot_multi_D_experiment(mse_opt, mse_unif, N, ALPHA, TRIALS)

  avg_opt_mses_k.append(avg_optimal_mse)
  avg_unif_mses_k.append(avg_uniform_mse)

print(f"Fix N = {N}, alpha = {ALPHA}, d = {D}.")
mse_k_df = pd.DataFrame({
    "k": ks,
    "Optimal": avg_opt_mses_k,
    "Uniform": avg_unif_mses_k,
    "Uniform - Optimal": [unif - opt for unif, opt in zip(avg_unif_mses_k, avg_opt_mses_k)],
})
print(mse_k_df)

Fix N = 1000, alpha = 0.5, d = 100.
     k   Optimal   Uniform  Uniform - Optimal
0    1  0.049425  1.418029           1.368603
1   50  0.057078  1.204203           1.147124
2   89  0.057876  1.268760           1.210883
3  100  0.058429  1.297006           1.238577
